In [1]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# WORKING DIRECTORY AND SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_6.root"
tree_name = "t"

selected_volumes = [400, 430, 445, 460, 490, 4901]
selected_pdg = 22

excel_file = "lung_TissueTumor_6_all_gamma_data.xlsx"

print("Working directory:", os.getcwd())
print("Input file:", os.path.abspath(file_path))

# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())

# =====================================================
# SAFE VECTOR READER
# =====================================================
def safe_value(vector, index, default=np.nan):
    """
    Return vector[index] when available.
    Otherwise return NaN.
    """
    if index < len(vector):
        return vector[index]
    return default

# =====================================================
# STORAGE
# =====================================================
volume_data = {
    volume: [] for volume in selected_volumes
}

combined_data = []

column_order = [
    "tree_entry",
    "vector_index",
    "vlm",
    "pdg",
    "pro",
    "stp",
    "trk",
    "k",
    "et",
    "de",
    "x",
    "y",
    "z",
    "px",
    "py",
    "pz"
]

# =====================================================
# LOOP THROUGH ROOT DATA
# =====================================================
for tree_entry, event in enumerate(tree):

    # Use pdg as the principal record length
    number_of_records = len(event.pdg)

    for vector_index in range(number_of_records):

        pdg_value = int(event.pdg[vector_index])

        # vlm must exist for this record
        if vector_index >= len(event.vlm):
            continue

        volume_value = int(event.vlm[vector_index])

        # Keep only gammas
        if pdg_value != selected_pdg:
            continue

        # Keep only detector volumes
        if volume_value not in selected_volumes:
            continue

        row = {
            "tree_entry": tree_entry,
            "vector_index": vector_index,

            "vlm": volume_value,
            "pdg": pdg_value,

            "pro": int(safe_value(event.pro, vector_index, -1)),
            "stp": int(safe_value(event.stp, vector_index, -1)),
             "trk": int(
                safe_value(event.trk, vector_index, -1)
                if hasattr(event, "trk")
                else -1
                        ),
            "k": float(safe_value(event.k, vector_index)),
            "et": float(safe_value(event.et, vector_index)),
            "de": float(safe_value(event.de, vector_index)),

            "x": float(safe_value(event.x, vector_index)),
            "y": float(safe_value(event.y, vector_index)),
            "z": float(safe_value(event.z, vector_index)),

            "px": float(safe_value(event.px, vector_index)),
            "py": float(safe_value(event.py, vector_index)),
            "pz": float(safe_value(event.pz, vector_index))
        }

        volume_data[volume_value].append(row)
        combined_data.append(row)

# =====================================================
# CREATE DATAFRAMES
# =====================================================
dataframes = {
    volume: pd.DataFrame(
        volume_data[volume],
        columns=column_order
    )
    for volume in selected_volumes
}

df_all = pd.DataFrame(
    combined_data,
    columns=column_order
)

# =====================================================
# WRITE EXCEL WORKBOOK
# =====================================================
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    for volume in selected_volumes:
        dataframes[volume].to_excel(
            writer,
            sheet_name=f"Volume_{volume}",
            index=False
        )

    df_all.to_excel(
        writer,
        sheet_name="All_Volumes",
        index=False
    )

# =====================================================
# CONFIRM OUTPUT
# =====================================================
print("\nRecords saved:")

for volume in selected_volumes:
    print(f"Volume {volume}: {len(dataframes[volume])}")

print("All volumes:", len(df_all))

print("\nExcel file saved at:")
print(os.path.abspath(excel_file))

print("File exists:", os.path.exists(excel_file))

if os.path.exists(excel_file):
    print("File size:", os.path.getsize(excel_file), "bytes")

display(df_all.head(10))

root_file.Close()

Working directory: /root/geant4/detector/Lung_ICRP
Input file: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_6.root
Total TTree entries: 100000

Records saved:
Volume 400: 133944
Volume 430: 1135
Volume 445: 599
Volume 460: 317
Volume 490: 212
Volume 4901: 222
All volumes: 136429

Excel file saved at:
/root/geant4/detector/Lung_ICRP/lung_TissueTumor_6_all_gamma_data.xlsx
File exists: True
File size: 22068146 bytes


,tree_entry,vector_index,vlm,pdg,pro,stp,trk,k,et,de,x,y,z,px,py,pz
0,0,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000
1,1,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000
2,3,6,400,22,2013,6,1,444.946943,0.0,0.0,0.000000,-45.053782,0.000000,0.577806,-0.623453,0.526732
3,3,7,400,22,1092,7,1,444.946943,0.0,0.0,7.390139,-53.027737,6.736902,0.577806,-0.623453,0.526732
4,4,6,400,22,2013,6,1,585.326059,0.0,0.0,0.000000,-47.267220,0.000000,0.298444,-0.898886,-0.320836
5,4,7,400,22,1092,7,1,585.326059,0.0,0.0,2.567401,-55.000000,-2.760030,0.298444,-0.898886,-0.320836
6,5,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000
7,6,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000
8,7,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000
9,8,6,400,22,1092,6,1,662.000000,NaN,0.0,0.000000,-55.000000,0.000000,0.000000,-1.000000,0.000000


In [4]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# WORKING DIRECTORY AND SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_445.root"
tree_name = "t"

selected_volumes = [2, 3, 445]
selected_pdg = 22

excel_file = "lung_TissueTumor_445_all_gamma_data.xlsx"

print("Working directory:", os.getcwd())
print("Input file:", os.path.abspath(file_path))

# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())

# =====================================================
# SAFE VECTOR READER
# =====================================================
def safe_value(vector, index, default=np.nan):
    """
    Return vector[index] when available.
    Otherwise return NaN.
    """
    if index < len(vector):
        return vector[index]
    return default

# =====================================================
# STORAGE
# =====================================================
volume_data = {
    volume: [] for volume in selected_volumes
}

combined_data = []

column_order = [
    "tree_entry",
    "vector_index",
    "vlm",
    "pdg",
    "pro",
    "stp",
    "trk",
    "k",
    "et",
    "de",
    "x",
    "y",
    "z",
    "px",
    "py",
    "pz"
]

# =====================================================
# LOOP THROUGH ROOT DATA
# =====================================================
for tree_entry, event in enumerate(tree):

    # Use pdg as the principal record length
    number_of_records = len(event.pdg)

    for vector_index in range(number_of_records):

        pdg_value = int(event.pdg[vector_index])

        # vlm must exist for this record
        if vector_index >= len(event.vlm):
            continue

        volume_value = int(event.vlm[vector_index])

        # Keep only gammas
        if pdg_value != selected_pdg:
            continue

        # Keep only detector volumes
        if volume_value not in selected_volumes:
            continue

        row = {
            "tree_entry": tree_entry,
            "vector_index": vector_index,

            "vlm": volume_value,
            "pdg": pdg_value,

            "pro": int(safe_value(event.pro, vector_index, -1)),
            "stp": int(safe_value(event.stp, vector_index, -1)),
             "trk": int(
                safe_value(event.trk, vector_index, -1)
                if hasattr(event, "trk")
                else -1
                        ),
            "k": float(safe_value(event.k, vector_index)),
            "et": float(safe_value(event.et, vector_index)),
            "de": float(safe_value(event.de, vector_index)),

            "x": float(safe_value(event.x, vector_index)),
            "y": float(safe_value(event.y, vector_index)),
            "z": float(safe_value(event.z, vector_index)),

            "px": float(safe_value(event.px, vector_index)),
            "py": float(safe_value(event.py, vector_index)),
            "pz": float(safe_value(event.pz, vector_index))
        }

        volume_data[volume_value].append(row)
        combined_data.append(row)

# =====================================================
# CREATE DATAFRAMES
# =====================================================
dataframes = {
    volume: pd.DataFrame(
        volume_data[volume],
        columns=column_order
    )
    for volume in selected_volumes
}

df_all = pd.DataFrame(
    combined_data,
    columns=column_order
)

# =====================================================
# WRITE EXCEL WORKBOOK
# =====================================================
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    for volume in selected_volumes:
        dataframes[volume].to_excel(
            writer,
            sheet_name=f"Volume_{volume}",
            index=False
        )

    df_all.to_excel(
        writer,
        sheet_name="All_Volumes",
        index=False
    )

# =====================================================
# CONFIRM OUTPUT
# =====================================================
print("\nRecords saved:")

for volume in selected_volumes:
    print(f"Volume {volume}: {len(dataframes[volume])}")

print("All volumes:", len(df_all))

print("\nExcel file saved at:")
print(os.path.abspath(excel_file))

print("File exists:", os.path.exists(excel_file))

if os.path.exists(excel_file):
    print("File size:", os.path.getsize(excel_file), "bytes")

display(df_all.head(10))

root_file.Close()

Working directory: /root/geant4/detector/Lung_ICRP
Input file: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_445.root
Total TTree entries: 100000

Records saved:
Volume 2: 202589
Volume 3: 103347
Volume 445: 187
All volumes: 306123

Excel file saved at:
/root/geant4/detector/Lung_ICRP/lung_TissueTumor_445_all_gamma_data.xlsx
File exists: True
File size: 33702830 bytes


,tree_entry,vector_index,vlm,pdg,pro,stp,trk,k,et,de,x,y,z,px,py,pz
0,0,2,2,22,1092,2,1,662.0,NaN,0.0,0.0,2.5,0.0,0.0,-1.0,0.0
1,0,3,3,22,1092,3,1,662.0,NaN,0.0,0.0,-2.5,0.0,0.0,-1.0,0.0
2,0,4,2,22,1092,4,1,662.0,NaN,0.0,0.0,-10.0,0.0,0.0,-1.0,0.0
3,1,2,2,22,1092,2,1,662.0,NaN,0.0,0.0,2.5,0.0,0.0,-1.0,0.0
4,1,3,3,22,1092,3,1,662.0,NaN,0.0,0.0,-2.5,0.0,0.0,-1.0,0.0
5,1,4,2,22,1092,4,1,662.0,NaN,0.0,0.0,-10.0,0.0,0.0,-1.0,0.0
6,2,2,2,22,1092,2,1,662.0,NaN,0.0,0.0,2.5,0.0,0.0,-1.0,0.0
7,2,3,3,22,1092,3,1,662.0,NaN,0.0,0.0,-2.5,0.0,0.0,-1.0,0.0
8,2,4,2,22,1092,4,1,662.0,NaN,0.0,0.0,-10.0,0.0,0.0,-1.0,0.0
9,3,2,2,22,1092,2,1,662.0,NaN,0.0,0.0,2.5,0.0,0.0,-1.0,0.0
